# Airbnb Price Prediction Using Machine Learning

This notebook is a beginner-friendly regression project. We will use a public New York City Airbnb dataset to predict listing prices.

Main steps:

1. Load dataset
2. Clean data
3. Explore data using visualizations
4. Prepare features and target
5. Train Linear Regression, Ridge Regression, and Random Forest
6. Evaluate models using MAE, RMSE, and R² Score
7. Study feature importance
8. Save the final project files for deployment

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_columns', 100)
print('Libraries imported successfully')

## 2. Load Public Dataset

We use a public New York City Airbnb dataset available as a CSV file.

In [ ]:
url = 'https://raw.githubusercontent.com/ManarOmar/New-York-Airbnb-2019/master/AB_NYC_2019.csv'
df = pd.read_csv(url)

print('Dataset shape:', df.shape)
df.head()

## 3. Basic Dataset Understanding

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
print('Missing prices:', df['price'].isnull().sum())
print('Non-missing prices:', df['price'].notnull().sum())
df['price'].describe()

## 4. Select Useful Features

For this educational project, we use practical features related to location, room type, reviews, host activity, and availability.

In [ ]:
selected_columns = [
    'price',
    'neighbourhood_group',
    'neighbourhood',
    'latitude',
    'longitude',
    'room_type',
    'minimum_nights',
    'number_of_reviews',
    'reviews_per_month',
    'calculated_host_listings_count',
    'availability_365'
]

data = df[selected_columns].copy()
print('Selected data shape:', data.shape)
data.head()

## 5. Data Cleaning

`reviews_per_month` is missing for listings with no reviews, so we replace missing values with 0. We also remove unrealistic prices and extreme minimum-night values.

In [ ]:
data['reviews_per_month'] = data['reviews_per_month'].fillna(0)
data = data.dropna()

data = data[(data['price'] >= 10) & (data['price'] <= 1000)]
data = data[data['minimum_nights'] <= 365]

data.isnull().sum()

In [ ]:
print('Cleaned data shape:', data.shape)
data['price'].describe()

## 6. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data['price'], bins=50, kde=True)
plt.title('Distribution of Airbnb Prices')
plt.xlabel('Price')
plt.ylabel('Number of Listings')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=data, x='room_type', y='price')
plt.title('Room Type vs Price')
plt.xlabel('Room Type')
plt.ylabel('Price')
plt.xticks(rotation=30)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=data, x='neighbourhood_group', y='price')
plt.title('Neighbourhood Group vs Price')
plt.xlabel('Neighbourhood Group')
plt.ylabel('Price')
plt.show()

In [ ]:
numeric_data = data.select_dtypes(include=['int64', 'float64'])
plt.figure(figsize=(10, 7))
sns.heatmap(numeric_data.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

## 7. Prepare Data for Machine Learning

In [ ]:
X = data.drop('price', axis=1)
y = data['price']

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training data:', X_train.shape)
print('Testing data:', X_test.shape)

## 8. Build Preprocessing Pipeline

Numerical features are imputed and scaled. Categorical features are imputed and one-hot encoded.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

## 9. Train Regression Models

We train three models and compare them using MAE, RMSE, and R² Score. We use log transformation on the target because price data is usually skewed.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(
        n_estimators=120,
        max_depth=18,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

results = []
trained_models = {}

for model_name, regressor in models.items():
    print(f'Training {model_name}...')
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', regressor)
    ])
    model = TransformedTargetRegressor(
        regressor=pipeline,
        func=np.log1p,
        inverse_func=np.expm1
    )
    model.fit(X_train, y_train)
    y_pred = np.maximum(model.predict(X_test), 0)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    results.append({'Model': model_name, 'MAE': mae, 'RMSE': rmse, 'R2 Score': r2})
    trained_models[model_name] = model

results_df = pd.DataFrame(results).sort_values(by='RMSE')
results_df

## 10. Visualize Model Comparison

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=results_df, x='Model', y='RMSE')
plt.title('Model Comparison Based on RMSE')
plt.ylabel('RMSE')
plt.xticks(rotation=30)
plt.show()

## 11. Select Best Model and Evaluate Predictions

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]

print('Best model:', best_model_name)

y_pred_best = np.maximum(best_model.predict(X_test), 0)

In [ ]:
plt.figure(figsize=(7, 7))
sns.scatterplot(x=y_test, y=y_pred_best, alpha=0.4)
plt.plot([0, 1000], [0, 1000], color='red', linestyle='--')
plt.title(f'Actual vs Predicted Prices - {best_model_name}')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.show()

In [ ]:
residuals = y_test - y_pred_best

plt.figure(figsize=(10, 5))
sns.histplot(residuals, bins=50, kde=True)
plt.title('Residual Distribution')
plt.xlabel('Residual = Actual Price - Predicted Price')
plt.ylabel('Count')
plt.show()

## 12. Feature Importance

In [ ]:
fitted_pipeline = best_model.regressor_
preprocessor_fitted = fitted_pipeline.named_steps['preprocessor']
regressor_fitted = fitted_pipeline.named_steps['regressor']

feature_names = preprocessor_fitted.get_feature_names_out()

if hasattr(regressor_fitted, 'feature_importances_'):
    importances = regressor_fitted.feature_importances_
else:
    importances = np.abs(regressor_fitted.coef_).ravel()

feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

feature_importance_df.head(20)

In [ ]:
plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importance_df.head(20), x='Importance', y='Feature')
plt.title('Top 20 Important Features')
plt.show()

## 13. Sample Predictions

In [ ]:
sample_results = X_test.copy()
sample_results['Actual Price'] = y_test
sample_results['Predicted Price'] = y_pred_best
sample_results['Absolute Error'] = abs(sample_results['Actual Price'] - sample_results['Predicted Price'])

sample_results[[
    'neighbourhood_group', 'neighbourhood', 'room_type',
    'minimum_nights', 'number_of_reviews',
    'Actual Price', 'Predicted Price', 'Absolute Error'
]].head(10)

## 14. Final Conclusion

In this project, we built an Airbnb price prediction system using machine learning regression models.

We used a public Airbnb dataset and selected important features such as neighbourhood group, neighbourhood, room type, location, minimum nights, number of reviews, reviews per month, host listing count, and yearly availability.

Three models were trained and compared:

1. Linear Regression
2. Ridge Regression
3. Random Forest Regressor

The final model can be deployed using the included Streamlit app in this repository.